# 🔌 VSCode – Google Colab Bağlantısı

Bu notebook, Google Colab ortamına bir SSH sunucusu kurarak **VS Code Remote-SSH** eklentisi üzerinden bağlanmanızı sağlar.

## Gereksinimler
- VS Code yüklü olmalı
- [Remote - SSH](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh) eklentisi yüklü olmalı

## Adımlar
1. Bu notebook'un tüm hücrelerini sırayla çalıştırın.
2. Son hücrede çıkan bağlantı bilgilerini not alın.
3. VS Code'da **Remote-SSH: Connect to Host...** komutunu kullanın.

In [ ]:
# Hücre 1: Gerekli paketleri kur
import subprocess

print("SSH sunucusu ve cloudflared kuruluyor...")

# openssh-server kur
subprocess.run(["apt-get", "install", "-y", "-q", "openssh-server"], check=True)

# cloudflared indir
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

print("Kurulum tamamlandı.")

In [ ]:
# Hücre 2: SSH kullanıcısı oluştur ve sunucuyu başlat
import subprocess

# İstediğiniz şifreyi buraya yazın
SSH_USER = "colab"
SSH_PASSWORD = "colab1234"

# sudo erişimli yeni kullanıcı oluştur (root girişi yerine)
subprocess.run(["useradd", "-m", "-s", "/bin/bash", SSH_USER], check=False)
subprocess.run(f"echo {SSH_USER}:{SSH_PASSWORD} | chpasswd", shell=True, check=True)
subprocess.run(["usermod", "-aG", "sudo", SSH_USER], check=True)

# SSH yapılandırması (root girişi devre dışı)
sshd_config = """
PermitRootLogin no
PasswordAuthentication yes
Port 22
"""
with open("/etc/ssh/sshd_config", "a") as f:
    f.write(sshd_config)

# SSH host anahtarlarını oluştur
subprocess.run(["ssh-keygen", "-A"], check=True)

# SSH servisini başlat
subprocess.run(["service", "ssh", "start"], check=True)

print(f"SSH sunucusu başlatıldı. Kullanıcı: {SSH_USER} | Şifre: {SSH_PASSWORD}")

In [ ]:
# Hücre 3: Cloudflare tüneli başlat ve bağlantı bilgilerini göster
import subprocess, threading, re

tunnel_url = None
proc = None

def start_tunnel():
    global tunnel_url, proc
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "tcp://localhost:22"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        match = re.search(r"(https://[\w.-]+\.trycloudflare\.com)", line)
        if match:
            tunnel_url = match.group(1)
            hostname = tunnel_url.replace("https://", "")
            print("\n" + "="*60)
            print("✅ Tünel hazır! VS Code bağlantı bilgileri:")
            print("="*60)
            print(f"  Host (HostName): {hostname}")
            print(f"  Kullanıcı adı : {SSH_USER}")
            print(f"  Şifre         : {SSH_PASSWORD}")
            print(f"  Port           : 22")
            print()
            print("📝 ~/.ssh/config dosyanıza ekleyin:")
            print("-"*60)
            print(f"Host colab")
            print(f"    HostName {hostname}")
            print(f"    User {SSH_USER}")
            print(f"    Port 22")
            print(f"    ProxyCommand cloudflared access ssh --hostname %h")
            print("-"*60)
            print()
            print("🚀 VS Code'da 'Remote-SSH: Connect to Host...' komutunu")
            print("   çalıştırıp 'colab' yazın ve Enter'a basın.")
            print("="*60)
            break

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()

print("Tünel başlatılıyor, lütfen bekleyin...")
t.join(timeout=60)
if tunnel_url is None:
    print("Tünel URL'si alınamadı. Çıktıyı kontrol edin.")

## 🔧 VS Code SSH Config Örneği

Yukarıdaki hücre çalıştıktan sonra, yerel makinenizdeki `~/.ssh/config` dosyasına aşağıdaki bloğu ekleyin:

```
Host colab
    HostName <tünelden-gelen-hostname>
    User colab
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h
```

> ⚠️ **Not:** Colab oturumu kapandığında tünel de kapanır. Her yeni Colab oturumunda bu notebook'u yeniden çalıştırmanız gerekir.